In [43]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import copy
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from scipy.optimize import least_squares

from inverse_pinn_model import InversePINN

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [44]:
# Same setting as before
device = torch.device("cpu")
dtype = torch.float64

TRAIN_POOL_SIZE = 200

data_sizes = [
    2,
    30,
    50,
    100,
    120,
    150,
    200
]
N_f = 2000
N_terminal = 500
N_boundary = 500

epsilon = 1e-4

hidden_width = 64
hidden_layers = 4

r_initial = 0.10
sigma_initial = 0.40

learning_rate = 1e-3
num_epochs = 30000

print_every = 5000
history_every = 100

lambda_data = 6.0
lambda_pde = 1.50
lambda_terminal = 1.0
lambda_boundary = 1.0


print("device =", device)
print("data_sizes =", data_sizes)

print()
print("N_f =", N_f)
print("N_terminal =", N_terminal)
print("N_boundary =", N_boundary)


device = cpu
data_sizes = [2, 30, 50, 100, 120, 150, 200]

N_f = 2000
N_terminal = 500
N_boundary = 500


In [45]:
# Load data
possible_paths = [
    Path("data") / "NDX_call_data.csv",
    Path("NDX_call_data.csv"),
    Path("NDX_call_data(1).csv")
]


data_path = None

for path in possible_paths:
    if path.exists():
        data_path = path
        break


if data_path is None:
    raise FileNotFoundError(
        "Could not find NDX_call_data.csv"
    )


print("Using data file:", data_path)

# read csv
raw_df = pd.read_csv(
    data_path
)

unnamed_columns = [
    col
    for col in raw_df.columns
    if str(col).startswith("Unnamed:")
]

if unnamed_columns:
    raw_df = raw_df.drop(
        columns=unnamed_columns
    )


raw_df = raw_df.reset_index(
    drop=True
)


# Save original row identity
raw_df["original_row"] = np.arange(
    len(raw_df)
)


# If market_price is not already: use bid/ask midpoint
if "market_price" not in raw_df.columns:

    if not {
        "best_bid",
        "best_offer"
    }.issubset(
        raw_df.columns
    ):

        raise KeyError(
            "Need market_price or "
            "best_bid and best_offer."
        )

    raw_df["market_price"] = (
        raw_df["best_bid"]
        + raw_df["best_offer"]
    ) / 2.0

# Strike Price

K = float(
    raw_df[
        "strike_price"
    ]
    .dropna()
    .iloc[0]
)

# Keep required columns 
real_df = raw_df[
    [
        "original_row",
        "current_time",
        "spot_price",
        "market_price"
    ]
].copy()

# remove invalid observations
real_df = (
    real_df
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .dropna()
)


real_df = real_df[
    (
        real_df["current_time"]
        >= 0.0
    )
    &
    (
        real_df["spot_price"]
        > 0.0
    )
    &
    (
        real_df["market_price"]
        >= 0.0
    )
].copy()


real_df = real_df.reset_index(
    drop=True
)


print(
    "Number of usable observations =",
    len(real_df)
)

print(
    "K =",
    K
)

Using data file: NDX_call_data.csv
Number of usable observations = 239
K = 15700.0


In [46]:
T = float(
    real_df[
        "current_time"
    ].max()
)


real_df["t"] = (
    T
    - real_df["current_time"]
)


# Physical stock price
real_df["S"] = (
    real_df["spot_price"]
)


# Physical option price
real_df["V"] = (
    real_df["market_price"]
)


print("T =", T)

print()

print(
    "t min =",
    real_df["t"].min()
)

print(
    "t max =",
    real_df["t"].max()
)

print()

print(
    "S min =",
    real_df["S"].min()
)

print(
    "S max =",
    real_df["S"].max()
)

print()

print(
    "V min =",
    real_df["V"].min()
)

print(
    "V max =",
    real_df["V"].max()
)


real_df.head()

T = 0.989041095890411

t min = 0.0
t max = 0.9863013698630138

S min = 11127.5703125
S max = 16573.33984375

V min = 0.2
V max = 1674.65


,original_row,current_time,spot_price,market_price,t,S,V
0,0,0.989041,14137.230469,511.00,0.000000,14137.230469,511.00
1,1,0.986301,14270.419922,544.25,0.002740,14270.419922,544.25
2,2,0.983562,14274.240234,530.00,0.005479,14274.240234,530.00
3,3,0.980822,14365.959961,572.45,0.008219,14365.959961,572.45
4,4,0.978082,14345.179688,557.45,0.010959,14345.179688,557.45


In [47]:
# Define physical domain
t_min = 0.0
t_max = T


S_min = 0.0


S_data_max = float(
    real_df[
        "S"
    ].max()
)

S_max = (
    10.0 * S_data_max
)


In [48]:
# Convert real observations to tensor
t_all = torch.tensor(
    real_df[
        "t"
    ].to_numpy(),
    dtype=dtype,
    device=device
).reshape(
    -1,
    1
)


S_all = torch.tensor(
    real_df[
        "S"
    ].to_numpy(),
    dtype=dtype,
    device=device
).reshape(
    -1,
    1
)


V_all = torch.tensor(
    real_df[
        "V"
    ].to_numpy(),
    dtype=dtype,
    device=device
).reshape(
    -1,
    1
)

# u_all for PINN and V_all for regression
u_all = (
    V_all / K
)


N_obs = len(
    real_df
)


print(
    "N observations =",
    N_obs
)

print()

print(
    "t_all shape =",
    t_all.shape
)

print(
    "S_all shape =",
    S_all.shape
)

print(
    "V_all shape =",
    V_all.shape
)

print(
    "u_all shape =",
    u_all.shape
)

N observations = 239

t_all shape = torch.Size([239, 1])
S_all shape = torch.Size([239, 1])
V_all shape = torch.Size([239, 1])
u_all shape = torch.Size([239, 1])


### Randomly select 200 train Points
- One random train/test split

- Randomly select 200 observations once.

In [49]:
permutation = np.random.permutation(
    N_obs
)

# Remaining observations = test set
train_pool_idx_np = permutation[
    :TRAIN_POOL_SIZE
]
test_idx_np = permutation[
    TRAIN_POOL_SIZE:
]

# Convert into torch
train_pool_idx = torch.tensor(
    train_pool_idx_np,
    dtype=torch.long,
    device=device
)

test_idx = torch.tensor(
    test_idx_np,
    dtype=torch.long,
    device=device
)

# PINN n_data subsets
sample_indices = {

    n_data:
        train_pool_idx[
            :n_data
        ]

    for n_data
    in data_sizes
}

# Fixed test set
t_test = t_all[
    test_idx
]

S_test = S_all[
    test_idx
]

V_test = V_all[
    test_idx
]

u_test = u_all[
    test_idx
]


In [50]:
# Safety checks
t_test = t_all[
    test_idx
]

S_test = S_all[
    test_idx
]

V_test = V_all[
    test_idx
]

u_test = u_all[
    test_idx
]



In [51]:
# PINN n = 200 must use exactly the saem data as Regression
assert torch.equal(
    sample_indices[200],
    train_pool_idx
)

# check for nested dataset 
for n_small, n_large in zip(
    data_sizes[:-1],
    data_sizes[1:]
):

    small_set = set(
        sample_indices[
            n_small
        ]
        .cpu()
        .numpy()
        .tolist()
    )

    large_set = set(
        sample_indices[
            n_large
        ]
        .cpu()
        .numpy()
        .tolist()
    )

    assert small_set.issubset(
        large_set
    )

print(
    "Total observations =",
    N_obs
)

print(
    "Training pool =",
    len(train_pool_idx)
)

print(
    "Test set =",
    len(test_idx)
)

Total observations = 239
Training pool = 200
Test set = 39


In [52]:
# save exact realization of the train/test split
split_df = real_df[
    [
        "original_row",
        "t",
        "S",
        "V"
    ]
].copy()


split_df[
    "split"
] = "test"


split_df.loc[
    train_pool_idx_np,
    "split"
] = "train_pool"

# position inside the randomly ordered 200-point pool
train_order = np.full(
    N_obs,
    np.nan
)


for order, idx in enumerate(
    train_pool_idx_np,
    start=1
):

    train_order[
        idx
    ] = order


split_df[
    "train_pool_order"
] = train_order


split_df.to_csv(
    "notebook11_train_test_split.csv",
    index=False
)



### Black-Scholes Eq.(5)

In [53]:
def black_scholes_call(
    t,
    S,
    K,
    T,
    r,
    sigma
):

    tau = (
        T - t
    )


    eps = 1e-12


    tau_safe = torch.clamp(
        tau,
        min=eps
    )


    S_safe = torch.clamp(
        S,
        min=eps
    )

# d1
    d1 = (

        torch.log(
            S_safe / K
        )

        + (
            r
            + 0.5
            * sigma ** 2
        )
        * tau_safe

    ) / (

        sigma
        * torch.sqrt(
            tau_safe
        )
    )

# d2
    d2 = (

        d1

        - sigma
        * torch.sqrt(
            tau_safe
        )
    )


    # Normal CDF
    N_d1 = (

        0.5

        * (
            1.0

            + torch.erf(
                d1
                / np.sqrt(2.0)
            )
        )
    )


    N_d2 = (

        0.5

        * (
            1.0

            + torch.erf(
                d2
                / np.sqrt(2.0)
            )
        )
    )

  # BS call price
    V = (

        S_safe
        * N_d1

        - K
        * torch.exp(
            -r
            * tau_safe
        )
        * N_d2
    )


    # Exact terminal payoff
    payoff = torch.relu(
        S - K
    )


    V = torch.where(
        tau <= eps,
        payoff,
        V
    )
    # payoff =  max(S-K, 0)

    V = torch.where(
        S <= eps,
        torch.zeros_like(V),
        V
    )


    return V


### Regression on the same 200 training points

In [54]:
# Fit r_tilde and sigma_tilde using the smae random 200 point training pool
t_reg_train = t_all[
    train_pool_idx
]

S_reg_train = S_all[
    train_pool_idx
]

V_reg_train = V_all[
    train_pool_idx
]

# regression residual function
# residual = BS - real makrket price
def regression_residuals(
    parameters
):

    r_value = float(
        parameters[0]
    )

    sigma_value = float(
        parameters[1]
    )


    with torch.no_grad():

        V_prediction = (
            black_scholes_call(

                t=t_reg_train,
                S=S_reg_train,

                K=K,
                T=T,

                r=r_value,
                sigma=sigma_value
            )
        )


    residuals = (

        V_prediction
        - V_reg_train
    )


    return (
        residuals
        .cpu()
        .numpy()
        .reshape(-1)
    )

# use nonlinear least squares calibration
# r>0, sigma > 0 
regression_fit = least_squares(

    regression_residuals,

    x0=np.array(
        [
            r_initial,
            sigma_initial
        ],
        dtype=float
    ),

    bounds=(
        np.array(
            [
                1e-8,
                1e-8
            ]
        ),

        np.array(
            [
                np.inf,
                np.inf
            ]
        )
    ),

    max_nfev=10000
)

# estimated regression parameters
r_tilde = float(
    regression_fit.x[0]
)


sigma_tilde = float(
    regression_fit.x[1]
)

# regression training MSE
# 1/Ntrain sum [C(S_trian, t_train; r_tilde, sig_tilde) - V_train]^2
with torch.no_grad():

    V_reg_train_prediction = (
        black_scholes_call(

            t=t_reg_train,
            S=S_reg_train,

            K=K,
            T=T,

            r=r_tilde,
            sigma=sigma_tilde
        )
    )


    regression_train_mse = (
        torch.mean(
            (
                V_reg_train_prediction
                - V_reg_train
            ) ** 2
        )
        .item()
    )
print("Regression completed")

print(
    f"r_tilde = {r_tilde:.10f}"
)

print(
    f"sigma_tilde = {sigma_tilde:.10f}"
)

print(
    f"Regression train MSE = "
    f"{regression_train_mse:.10f}"
)

print(
    "Optimization success =",
    regression_fit.success
)


Regression completed
r_tilde = 0.0107174140
sigma_tilde = 0.2070400695
Regression train MSE = 6809.9791859488
Optimization success = True


### MSE1: Regression

In [55]:
with torch.no_grad():

    V_regression_test = black_scholes_call(
        t=t_test,
        S=S_test,
        K=K,
        T=T,
        r=r_tilde,
        sigma=sigma_tilde
    )

    mse_1_regression = torch.mean(
        (
            V_regression_test
            - V_test
        ) ** 2
    ).item()


print(
    "MSE #1: Regression Eq.(5) test MSE =",
    mse_1_regression
)


regression_result_df = pd.DataFrame(
    {
        "r_tilde": [r_tilde],
        "sigma_tilde": [sigma_tilde],
        "train_mse": [regression_train_mse],
        "test_mse_eq5": [mse_1_regression]
    }
)


regression_result_df

MSE #1: Regression Eq.(5) test MSE = 5152.11285764772


,r_tilde,sigma_tilde,train_mse,test_mse_eq5
0,0.010717,0.20704,6809.979186,5152.112858


In [56]:
# BS PDE residual

def black_scholes_residual(
    model,
    t,
    S
):

    t_input = (
        t.clone()
        .detach()
        .requires_grad_(True)
    )


    S_input = (
        S.clone()
        .detach()
        .requires_grad_(True)
    )


    u = model(
        t_input,
        S_input
    )


    # du/dt
    u_t = torch.autograd.grad(

        outputs=u,
        inputs=t_input,

        grad_outputs=torch.ones_like(
            u
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    # du/dS
    u_S = torch.autograd.grad(

        outputs=u,
        inputs=S_input,

        grad_outputs=torch.ones_like(
            u
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    # d2u/dS2
    u_SS = torch.autograd.grad(

        outputs=u_S,
        inputs=S_input,

        grad_outputs=torch.ones_like(
            u_S
        ),

        create_graph=True,
        retain_graph=True

    )[0]


    residual = (

        u_t

        + 0.5
        * model.sigma ** 2
        * S_input ** 2
        * u_SS

        + model.r
        * S_input
        * u_S

        - model.r
        * u
    )


    return residual


### PINN PDE residual + loss

In [57]:
# PINN loss
def compute_losses(

    model,

    t_train,
    S_train,
    u_train,

    t_f,
    S_f,

    t_terminal,
    S_terminal,

    t_boundary,
    S_boundary_left,
    S_boundary_right
):

    # 1. Data loss

    u_data_prediction = model(
        t_train,
        S_train
    )


    loss_data = torch.mean(

        (
            u_data_prediction
            - u_train
        ) ** 2
    )


    # 2. PDE loss

    pde_prediction = (
        black_scholes_residual(
            model,
            t_f,
            S_f
        )
    )


    loss_pde = torch.mean(
        pde_prediction ** 2
    )


    # 3. Terminal loss
    #
    # V(T,S) = max(S-K,0)
    #
    # u(T,S) = V/K

    u_terminal_prediction = model(
        t_terminal,
        S_terminal
    )


    u_terminal_true = (

        torch.relu(
            S_terminal
            - K
        )

        / K
    )


    loss_terminal = torch.mean(

        (
            u_terminal_prediction
            - u_terminal_true
        ) ** 2
    )

    # 4. Left boundary
    #
    # V(t,0) = 0

    u_left_prediction = model(
        t_boundary,
        S_boundary_left
    )


    u_left_true = (
        torch.zeros_like(
            u_left_prediction
        )
    )


    loss_boundary_left = torch.mean(

        (
            u_left_prediction
            - u_left_true
        ) ** 2
    )


    # 5. Right boundary
    #
    # V approximately
    #
    # S - K exp[-r(T-t)]

    u_right_prediction = model(
        t_boundary,
        S_boundary_right
    )


    u_right_true = (

        S_boundary_right

        - K
        * torch.exp(

            -model.r

            * (
                T
                - t_boundary
            )
        )

    ) / K


    loss_boundary_right = torch.mean(

        (
            u_right_prediction
            - u_right_true
        ) ** 2
    )


    loss_boundary = (

        loss_boundary_left
        + loss_boundary_right
    )


    # Total loss

    total_loss = (

        lambda_data
        * loss_data

        + lambda_pde
        * loss_pde

        + lambda_terminal
        * loss_terminal

        + lambda_boundary
        * loss_boundary
    )


    return {

        "total":
            total_loss,

        "data":
            loss_data,

        "pde":
            loss_pde,

        "terminal":
            loss_terminal,

        "boundary":
            loss_boundary
    }

In [58]:
# shared physics points; no fixed random seed
# All seven PINNs use exactly the same physics points

# PDE collocation points
#
# t in [t_min, t_max]
# S in [S_min, S_max]

t_f = (

    t_min
    + epsilon

    + (
        t_max
        - t_min
        - 2.0 * epsilon
    )

    * torch.rand(
        N_f,
        1,
        dtype=dtype,
        device=device
    )
)


S_f = (

    S_min
    + epsilon

    + (
        S_max
        - S_min
        - 2.0 * epsilon
    )

    * torch.rand(
        N_f,
        1,
        dtype=dtype,
        device=device
    )
)

# Terminal-condition points
# t = T
# S in [0, S_max]
t_f = (

    t_min
    + epsilon

    + (
        t_max
        - t_min
        - 2.0 * epsilon
    )

    * torch.rand(
        N_f,
        1,
        dtype=dtype,
        device=device
    )
)


S_f = (

    S_min
    + epsilon

    + (
        S_max
        - S_min
        - 2.0 * epsilon
    )

    * torch.rand(
        N_f,
        1,
        dtype=dtype,
        device=device
    )
)

# Boundary-condition points
t_boundary = (

    t_min

    + (
        t_max
        - t_min
    )

    * torch.rand(
        N_boundary,
        1,
        dtype=dtype,
        device=device
    )
)


# Left boundary S = 0
S_boundary_left = torch.full(

    (
        N_boundary,
        1
    ),

    fill_value=S_min,

    dtype=dtype,
    device=device
)


# Right boundary S = new S_max
S_boundary_right = torch.full(

    (
        N_boundary,
        1
    ),

    fill_value=S_max,

    dtype=dtype,
    device=device
)

print()

print(
    "PDE t range:",
    float(
        t_f.min()
    ),
    "to",
    float(
        t_f.max()
    )
)

print(
    "PDE S range:",
    float(
        S_f.min()
    ),
    "to",
    float(
        S_f.max()
    )
)

print()

print(
    "Right boundary S =",
    float(
        S_boundary_right[0]
    )
)



PDE t range: 0.0006563335615043138 to 0.9887085715838526
PDE S range: 20.829420961379615 to 165732.24139671426

Right boundary S = 165733.3984375


In [59]:
# Create ONE random initialization
# Then every n_data model starts from this same state

base_model = InversePINN(

    t_min=t_min,
    t_max=t_max,

    S_min=S_min,
    S_max=S_max,

    hidden_width=hidden_width,
    hidden_layers=hidden_layers,

    r_initial=r_initial,
    sigma_initial=sigma_initial

).to(
    device=device,
    dtype=dtype
)


base_state_dict = copy.deepcopy(
    base_model.state_dict()
)
print(
    "Initial r =",
    base_model.r.item()
)

print(
    "Initial sigma =",
    base_model.sigma.item()
)


del base_model

# random 一次 initialization，然后7个experiment 用同一个

Initial r = 0.10000000000000002
Initial sigma = 0.4


In [61]:
# Save this random realization
t_terminal = torch.full(
    (N_terminal, 1),
    T,
    dtype=dtype,
    device=device
)

S_terminal = (
    S_min
    + (S_max - S_min)
    * torch.rand(
        N_terminal,
        1,
        dtype=dtype,
        device=device
    )
)

torch.save(

    {
        "train_pool_indices":
            train_pool_idx.cpu(),

        "test_indices":
            test_idx.cpu(),

        "t_f":
            t_f.cpu(),

        "S_f":
            S_f.cpu(),

        "t_terminal":
            t_terminal.cpu(),

        "S_terminal":
            S_terminal.cpu(),

        "t_boundary":
            t_boundary.cpu(),

        "S_boundary_left":
            S_boundary_left.cpu(),

        "S_boundary_right":
            S_boundary_right.cpu(),

        "base_state_dict":
            base_state_dict,

        "K":
            K,

        "T":
            T,

        "S_max":
            S_max
    },

    "notebook11_random_realization.pt"
)


print(
    "Saved: notebook11_random_realization.pt"
)

Saved: notebook11_random_realization.pt


In [63]:
# Train ONE PINN for a given n_data
def train_pinn_for_n_data(
    n_data
):

    start_time = time.time()


    # Select real training observations

    selected_idx = (
        sample_indices[
            n_data
        ]
    )


    t_train = t_all[
        selected_idx
    ]


    S_train = S_all[
        selected_idx
    ]


    V_train = V_all[
        selected_idx
    ]


    # PINN price transformation
    u_train = (
        V_train / K
    )


    # Fresh model
    #
    # Load identical shared initialization

    model = InversePINN(

        t_min=t_min,
        t_max=t_max,

        S_min=S_min,
        S_max=S_max,

        hidden_width=hidden_width,
        hidden_layers=hidden_layers,

        r_initial=r_initial,
        sigma_initial=sigma_initial

    ).to(
        device=device,
        dtype=dtype
    )


    model.load_state_dict(
        copy.deepcopy(
            base_state_dict
        )
    )


    # Optimizer

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate
    )


    # History

    history = {

        "epoch": [],

        "total_loss": [],
        "data_loss": [],
        "pde_loss": [],
        "terminal_loss": [],
        "boundary_loss": [],

        "r_estimate": [],
        "sigma_estimate": []
    }


    print()
    print(
        "=" * 70
    )

    print(
        f"Training PINN with n_data = {n_data}"
    )

    print(
        "=" * 70
    )


    # Training

    for epoch in range(
        1,
        num_epochs + 1
    ):

        model.train()


        optimizer.zero_grad(
            set_to_none=True
        )


        losses = compute_losses(

            model=model,

            t_train=t_train,
            S_train=S_train,
            u_train=u_train,

            t_f=t_f,
            S_f=S_f,

            t_terminal=t_terminal,
            S_terminal=S_terminal,

            t_boundary=t_boundary,
            S_boundary_left=S_boundary_left,
            S_boundary_right=S_boundary_right
        )


        losses[
            "total"
        ].backward()


        optimizer.step()


        # Save history

        if (
            epoch == 1

            or epoch
            % history_every
            == 0

            or epoch
            == num_epochs
        ):

            history[
                "epoch"
            ].append(
                epoch
            )


            history[
                "total_loss"
            ].append(
                losses[
                    "total"
                ].item()
            )


            history[
                "data_loss"
            ].append(
                losses[
                    "data"
                ].item()
            )


            history[
                "pde_loss"
            ].append(
                losses[
                    "pde"
                ].item()
            )


            history[
                "terminal_loss"
            ].append(
                losses[
                    "terminal"
                ].item()
            )


            history[
                "boundary_loss"
            ].append(
                losses[
                    "boundary"
                ].item()
            )


            history[
                "r_estimate"
            ].append(
                model.r.item()
            )


            history[
                "sigma_estimate"
            ].append(
                model.sigma.item()
            )


        # Print progress
        if (
            epoch == 1

            or epoch
            % print_every
            == 0

            or epoch
            == num_epochs
        ):

            print(

                f"Epoch {epoch:5d} | "

                f"Total: "
                f"{losses['total'].item():.6e} | "

                f"Data: "
                f"{losses['data'].item():.6e} | "

                f"PDE: "
                f"{losses['pde'].item():.6e} | "

                f"r: "
                f"{model.r.item():.8f} | "

                f"sigma: "
                f"{model.sigma.item():.8f}"
            )


    # Final PINN parameters

    r_pinn = (
        model.r.item()
    )


    sigma_pinn = (
        model.sigma.item()
    )

    # TEST SET

    model.eval()


    with torch.no_grad():

        # MSE #2
        #
        # NN itself predicts u = V/K
        # Convert back to physical V
        u_pinn_test = model(
            t_test,
            S_test
        )


        V_pinn_nn_test = (
            K
            * u_pinn_test
        )


        mse_2_pinn_nn = (
            torch.mean(

                (
                    V_pinn_nn_test
                    - V_test
                ) ** 2

            )
            .item()
        )


        # MSE #3
        #
        # Take r_PINN and sigma_PINN
        # and plug them into Eq.(5)

        V_pinn_eq5_test = (
            black_scholes_call(

                t=t_test,
                S=S_test,

                K=K,
                T=T,

                r=r_pinn,
                sigma=sigma_pinn
            )
        )


        mse_3_pinn_eq5 = (
            torch.mean(

                (
                    V_pinn_eq5_test
                    - V_test
                ) ** 2

            )
            .item()
        )


    runtime_minutes = (

        time.time()
        - start_time

    ) / 60.0


    # Result dictionary

    result = {

        "n_data":
            n_data,

        "r_tilde":
            r_tilde,

        "sigma_tilde":
            sigma_tilde,

        "r_pinn":
            r_pinn,

        "sigma_pinn":
            sigma_pinn,

        "mse_1_regression_eq5_test":
            mse_1_regression,

        "mse_2_pinn_nn_test":
            mse_2_pinn_nn,

        "mse_3_pinn_eq5_test":
            mse_3_pinn_eq5,

        "runtime_minutes":
            runtime_minutes
    }


    print()

    print(
        f"n_data = {n_data} completed"
    )

    print()

    print(
        f"r_PINN = "
        f"{r_pinn:.10f}"
    )

    print(
        f"sigma_PINN = "
        f"{sigma_pinn:.10f}"
    )

    print()

    print(
        "MSE #1 Regression Eq.(5) =",
        mse_1_regression
    )

    print(
        "MSE #2 PINN NN =",
        mse_2_pinn_nn
    )

    print(
        "MSE #3 PINN r,sigma -> Eq.(5) =",
        mse_3_pinn_eq5
    )

    print()

    print(
        f"Runtime = "
        f"{runtime_minutes:.2f} minutes"
    )

    print(
        "-" * 70
    )


    return (

        result,

        model,

        history,

        V_pinn_nn_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1),

        V_pinn_eq5_test
        .detach()
        .cpu()
        .numpy()
        .reshape(-1)
    )

In [65]:
# Train all PINNS
results = []

# Save test-set predictions
test_predictions_df = (
    real_df
    .iloc[
        test_idx_np
    ]
    [
        [
            "original_row",
            "t",
            "S",
            "V"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


# Regression test predictions
test_predictions_df[
    "regression_eq5"
] = (

    V_regression_test
    .detach()
    .cpu()
    .numpy()
    .reshape(-1)
)

# run every n_data
for n_data in data_sizes:

    (
        result_n,
        model_n,
        history_n,
        V_pinn_nn_test_n,
        V_pinn_eq5_test_n

    ) = train_pinn_for_n_data(
        n_data
    )


    results.append(
        result_n
    )


    # ========================================================
    # Save test predictions
    # ========================================================

    test_predictions_df[
        f"pinn_nn_n{n_data}"
    ] = V_pinn_nn_test_n


    test_predictions_df[
        f"pinn_eq5_n{n_data}"
    ] = V_pinn_eq5_test_n


    # Save training history

    pd.DataFrame(
        history_n
    ).to_csv(

        f"notebook11_history_n{n_data}.csv",

        index=False
    )


    # Save model checkpoint

    torch.save(

        {
            "n_data":
                n_data,

            "model_state_dict":
                model_n.state_dict(),

            "selected_indices":
                sample_indices[
                    n_data
                ].cpu(),

            "train_pool_indices":
                train_pool_idx.cpu(),

            "test_indices":
                test_idx.cpu(),

            "result":
                result_n,

            "K":
                K,

            "T":
                T,

            "t_min":
                t_min,

            "t_max":
                t_max,

            "S_min":
                S_min,

            "S_max":
                S_max,

            "S_data_max":
                S_data_max,

            "N_f":
                N_f,

            "N_terminal":
                N_terminal,

            "N_boundary":
                N_boundary,

            "hidden_width":
                hidden_width,

            "hidden_layers":
                hidden_layers,

            "learning_rate":
                learning_rate,

            "num_epochs":
                num_epochs,

            "lambda_data":
                lambda_data,

            "lambda_pde":
                lambda_pde,

            "lambda_terminal":
                lambda_terminal,

            "lambda_boundary":
                lambda_boundary
        },

        f"notebook11_pinn_n{n_data}.pt"
    )


    # Save summary after each completed n_data
    pd.DataFrame(
        results
    ).to_csv(

        "notebook11_test_mse_results.csv",

        index=False
    )


    del model_n


# Save all test predictions

test_predictions_df.to_csv(

    "notebook11_test_predictions.csv",

    index=False
)


results_df = pd.DataFrame(
    results
)


print()


Training PINN with n_data = 2
Epoch     1 | Total: 1.255672e+02 | Data: 2.204178e-02 | PDE: 3.569192e-02 | r: 0.09990488 | sigma: 0.40032979
Epoch  5000 | Total: 6.871342e-04 | Data: 1.319620e-05 | PDE: 1.978084e-04 | r: 0.07680551 | sigma: 0.27333838
Epoch 10000 | Total: 3.163476e-04 | Data: 1.127069e-06 | PDE: 5.069312e-05 | r: 0.06520956 | sigma: 0.21949078
Epoch 15000 | Total: 1.261136e-04 | Data: 5.294190e-06 | PDE: 8.788366e-06 | r: 0.05488484 | sigma: 0.13974343
Epoch 20000 | Total: 6.493216e-05 | Data: 1.088134e-06 | PDE: 5.215629e-06 | r: 0.05726897 | sigma: 0.09598117
Epoch 25000 | Total: 2.611170e-04 | Data: 2.757160e-05 | PDE: 5.963363e-06 | r: 0.06446585 | sigma: 0.08527509
Epoch 30000 | Total: 1.322238e-04 | Data: 1.142969e-05 | PDE: 3.245317e-06 | r: 0.07247806 | sigma: 0.08532411

n_data = 2 completed

r_PINN = 0.0724780600
sigma_PINN = 0.0853241094

MSE #1 Regression Eq.(5) = 5152.11285764772
MSE #2 PINN NN = 30778.21517449717
MSE #3 PINN r,sigma -> Eq.(5) = 42931.429

In [66]:
final_columns = [

    "n_data",

    "r_tilde",
    "sigma_tilde",

    "r_pinn",
    "sigma_pinn",

    "mse_1_regression_eq5_test",
    "mse_2_pinn_nn_test",
    "mse_3_pinn_eq5_test",

    "runtime_minutes"
]


final_results = (
    results_df[
        final_columns
    ]
    .copy()
)


final_results

,n_data,r_tilde,sigma_tilde,r_pinn,sigma_pinn,mse_1_regression_eq5_test,mse_2_pinn_nn_test,mse_3_pinn_eq5_test,runtime_minutes
0,2,0.010717,0.20704,0.072478,0.085324,5152.112858,30778.215174,42931.429304,11.671689
1,30,0.010717,0.20704,0.087599,0.049852,5152.112858,4355.249657,50621.391444,10.311486
2,50,0.010717,0.20704,0.089958,0.050964,5152.112858,3561.632951,45000.144393,9.737417
3,100,0.010717,0.20704,0.088670,0.048076,5152.112858,18921.534627,49865.422871,10.177424
4,120,0.010717,0.20704,0.087623,0.044216,5152.112858,4963.791406,55293.484302,10.169393
5,150,0.010717,0.20704,0.086143,0.046012,5152.112858,5918.117157,57175.413439,10.168811
6,200,0.010717,0.20704,0.086125,0.036721,5152.112858,5826.356650,65004.722039,10.185403


In [67]:
results_df[
    [
        "n_data",
        "r_pinn",
        "sigma_pinn",
        "mse_2_pinn_nn_test",
        "mse_3_pinn_eq5_test"
    ]
]

,n_data,r_pinn,sigma_pinn,mse_2_pinn_nn_test,mse_3_pinn_eq5_test
0,2,0.072478,0.085324,30778.215174,42931.429304
1,30,0.087599,0.049852,4355.249657,50621.391444
2,50,0.089958,0.050964,3561.632951,45000.144393
3,100,0.088670,0.048076,18921.534627,49865.422871
4,120,0.087623,0.044216,4963.791406,55293.484302
5,150,0.086143,0.046012,5918.117157,57175.413439
6,200,0.086125,0.036721,5826.356650,65004.722039


In [68]:
for n_data in data_sizes:

    history = pd.read_csv(
        f"notebook11_history_n{n_data}.csv"
    )

    print(
        "\nn_data =", n_data
    )

    print(
        history.iloc[-1][
            [
                "total_loss",
                "data_loss",
                "pde_loss",
                "terminal_loss",
                "boundary_loss",
                "r_estimate",
                "sigma_estimate"
            ]
        ]
    )

# r_PINN = 0.086 ~ 0.090
# sigma_PINN = 0.037 ~ 0.051


n_data = 2
total_loss        0.000132
data_loss         0.000011
pde_loss          0.000003
terminal_loss     0.000038
boundary_loss     0.000021
r_estimate        0.072478
sigma_estimate    0.085324
Name: 300, dtype: float64

n_data = 30
total_loss        1.707537e-04
data_loss         2.396761e-05
pde_loss          7.225296e-06
terminal_loss     1.576481e-05
boundary_loss     3.452410e-07
r_estimate        8.759934e-02
sigma_estimate    4.985204e-02
Name: 300, dtype: float64

n_data = 50
total_loss        0.000146
data_loss         0.000016
pde_loss          0.000015
terminal_loss     0.000016
boundary_loss     0.000010
r_estimate        0.089958
sigma_estimate    0.050964
Name: 300, dtype: float64

n_data = 100
total_loss        0.000489
data_loss         0.000048
pde_loss          0.000019
terminal_loss     0.000047
boundary_loss     0.000125
r_estimate        0.088670
sigma_estimate    0.048076
Name: 300, dtype: float64

n_data = 120
total_loss        0.000436
data_loss         0

In [69]:
comparison = results_df[
    [
        "n_data",
        "r_tilde",
        "sigma_tilde",
        "r_pinn",
        "sigma_pinn",
        "mse_1_regression_eq5_test",
        "mse_2_pinn_nn_test",
        "mse_3_pinn_eq5_test"
    ]
]

comparison

,n_data,r_tilde,sigma_tilde,r_pinn,sigma_pinn,mse_1_regression_eq5_test,mse_2_pinn_nn_test,mse_3_pinn_eq5_test
0,2,0.010717,0.20704,0.072478,0.085324,5152.112858,30778.215174,42931.429304
1,30,0.010717,0.20704,0.087599,0.049852,5152.112858,4355.249657,50621.391444
2,50,0.010717,0.20704,0.089958,0.050964,5152.112858,3561.632951,45000.144393
3,100,0.010717,0.20704,0.088670,0.048076,5152.112858,18921.534627,49865.422871
4,120,0.010717,0.20704,0.087623,0.044216,5152.112858,4963.791406,55293.484302
5,150,0.010717,0.20704,0.086143,0.046012,5152.112858,5918.117157,57175.413439
6,200,0.010717,0.20704,0.086125,0.036721,5152.112858,5826.356650,65004.722039
